# KMFM Rebuild / LASSF — Colab workflow

本 notebook 从固定划分、训练、测试运行到多 seed 汇总。它不会生成或修改论文数字。请先把真实数据与本工程放到 Google Drive。第一次只运行 1–2 个 seed 的 pilot，确认 artifacts 和指标闭环后再扩展。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, json, subprocess, copy, tempfile, shutil

COLAB_ROOT = Path('/content/drive/MyDrive/Colab')
UNSUPERVISED_ROOT = COLAB_ROOT / 'Unsupervised'
PROJECT_ROOT = UNSUPERVISED_ROOT / 'KMFM'
DATA_ROOT = COLAB_ROOT / 'Datasets'
SPLIT_ROOT = PROJECT_ROOT / 'splits'
RESULT_ROOT = PROJECT_ROOT / 'results'
REPORT_ROOT = PROJECT_ROOT / 'reports'
REPO_URL = os.environ.get('KMFM_REPO_URL', 'https://github.com/chenzizi07/KMFM.git')

def _colab_github_token():
    try:
        from google.colab import userdata
    except ImportError:
        return None
    for name in ('GITHUB_TOKEN', 'GH_TOKEN', 'GITHUB_PAT', 'SASM_MAMBA_GITHUB_TOKEN', 'SASM_MAMBA_PAT'):
        try:
            value = userdata.get(name)
        except Exception:
            continue
        if value:
            return str(value)
    return None

def _git(args, cwd):
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    env['GIT_SSH_COMMAND'] = 'ssh -o BatchMode=yes'
    token = _colab_github_token()
    askpass = None
    if token:
        askpass_fd, askpass_name = tempfile.mkstemp(prefix='kmfm_askpass_', suffix='.sh', dir='/tmp')
        os.close(askpass_fd)
        askpass = Path(askpass_name)
        askpass.write_text("#!/bin/sh\ncase \"$1\" in\n  *Username*|*username*) printf '%s\\n' 'x-access-token' ;;\n  *) printf '%s\\n' \"$KMFM_GIT_TOKEN\" ;;\nesac\n")
        askpass.chmod(0o700)
        env['GIT_ASKPASS'] = str(askpass)
        env['KMFM_GIT_TOKEN'] = token
    try:
        return subprocess.run(['git', *args], cwd=cwd, env=env, check=True)
    finally:
        if askpass is not None: askpass.unlink(missing_ok=True)

UNSUPERVISED_ROOT.mkdir(parents=True, exist_ok=True)
if not (PROJECT_ROOT / '.git').exists():
    if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
        raise RuntimeError(f'Non-git project directory exists; refusing to overwrite: {PROJECT_ROOT}')
    _git(['clone', '--', REPO_URL, str(PROJECT_ROOT)], cwd=UNSUPERVISED_ROOT)
else:
    subprocess.run(['bash', str(PROJECT_ROOT / 'scripts' / 'colab_update.sh')], cwd=PROJECT_ROOT, check=True, env={**os.environ, 'REPO_URL': REPO_URL, 'PROJECT_DIR': str(PROJECT_ROOT), 'PROJECT_NAME': 'KMFM', 'DRIVE_ROOT': str(UNSUPERVISED_ROOT)})
os.chdir(PROJECT_ROOT)
print('Project:', PROJECT_ROOT)
print('Data:', DATA_ROOT)
print('Repository:', REPO_URL)

In [ ]:
!python -m pip -q install -r requirements-colab.txt
!python -m pip -q install -e .
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU before formal training.'

In [ ]:
!python scripts/smoke_test.py

## Dataset settings

先选择一个数据集。若你的 MAT key 不同，请修改。设置为 `None` 时加载器会自动选择最大的 3-D/2-D 数组，但正式实验建议明确填写。

In [ ]:
DATASETS = {
    'pavia_university': dict(data='PaviaU.mat', gt='PaviaU_gt.mat', data_key='paviaU', gt_key='paviaU_gt'),
    'houston2013': dict(data='Houston_data.mat', gt='Houston_gt.mat', data_key='hsi', gt_key='groundT'),
    'ksc': dict(data='KSC.mat', gt='KSC_gt.mat', data_key='KSC', gt_key='KSC_gt'),
    'botswana': dict(data='Botswana.mat', gt='Botswana_gt.mat', data_key='Botswana', gt_key='Botswana_gt'),
}
DATASET = 'pavia_university'
spec = DATASETS[DATASET]
data_path = DATA_ROOT / spec['data']
gt_path = DATA_ROOT / spec['gt']
assert data_path.exists(), data_path
assert gt_path.exists(), gt_path
print(DATASET, data_path, gt_path)

## Generate immutable splits

Pilot 使用 seeds 0 和 1。正式 RP 建议 10 seeds；SB 先做 5 seeds，再根据类别空间分布和 Colab 配额扩展。已有 split 不会自动覆盖。

In [ ]:
PILOT = True
SEEDS = [0, 1] if PILOT else list(range(10))
PROTOCOLS = ['random_pixel', 'spatial_block']

for protocol in PROTOCOLS:
    for seed in SEEDS:
        split_path = SPLIT_ROOT / DATASET / protocol / f'seed_{seed}.npz'
        if split_path.exists():
            print('Using existing split:', split_path)
            continue
        cmd = [
            'python', 'scripts/make_split.py',
            '--data-path', str(data_path), '--gt-path', str(gt_path),
            '--protocol', protocol, '--seed', str(seed),
            '--train-per-class', '30', '--val-per-class', '10',
            '--output', str(split_path),
        ]
        if spec.get('data_key'): cmd += ['--data-key', spec['data_key']]
        if spec.get('gt_key'): cmd += ['--gt-key', spec['gt_key']]
        if protocol == 'spatial_block':
            cmd += ['--min-test-per-class', '20', '--block-size', '32', '--buffer-pixels', '3', '--trials', '256']
        subprocess.run(cmd, check=True)

## Pilot training matrix

修改 `EXPERIMENT` 后运行。相同 experiment/dataset/protocol/model/seed 不会被覆盖。Pilot 默认为 50 epochs；正式实验改为 200，并保留 early stopping。

In [ ]:
from kmfm.engine import resolve_run_dir, run_experiment

EXPERIMENT = 'pilot_v1'  # rerun a completed matrix with a new name
VARIANTS = [
    dict(name='lassf_mlp_concat_h64', spectral='mlp', fusion='concat'),
    dict(name='lassf_conv1d_concat_h64', spectral='conv1d', fusion='concat'),
    dict(name='lassf_conv1d_gate_h64', spectral='conv1d', fusion='gate'),
    dict(name='lassf_conv1d_reliability_h64', spectral='conv1d', fusion='reliability'),
]
if not PILOT:
    VARIANTS += [
        dict(name='lassf_conv1d_spatial_only_h64', spectral='conv1d', fusion='spatial_only'),
        dict(name='lassf_conv1d_spectral_only_h64', spectral='conv1d', fusion='spectral_only'),
    ]

base = {
    'seed': 0,
    'data': {
        'name': DATASET, 'data_path': str(data_path), 'gt_path': str(gt_path),
        'data_key': spec.get('data_key'), 'gt_key': spec.get('gt_key'), 'zscore_clip': 8.0,
    },
    'protocol': {'name': 'spatial_block', 'split_path': ''},
    'model': {'name': '', 'hidden_dim': 64, 'spectral': 'conv1d', 'fusion': 'reliability', 'dropout': 0.1},
    'training': {
        'patch_size': 11, 'batch_size': 64, 'num_workers': 2,
        'epochs': 50 if PILOT else 200, 'patience': 15 if PILOT else 30,
        'lr': 3e-4, 'weight_decay': 1e-4, 'aux_weight': 0.2,
        'amp': True, 'deterministic': True,
    },
    'output': {'root': str(RESULT_ROOT), 'experiment': EXPERIMENT},
}

for protocol in PROTOCOLS:
    for variant in VARIANTS:
        for seed in SEEDS:
            cfg = copy.deepcopy(base)
            cfg['seed'] = seed
            cfg['protocol'] = {
                'name': protocol,
                'split_path': str(SPLIT_ROOT / DATASET / protocol / f'seed_{seed}.npz'),
            }
            cfg['model'].update(variant)
            run_dir = resolve_run_dir(cfg)
            status_path = run_dir / 'status.json'
            status = json.loads(status_path.read_text()).get('state') if status_path.exists() else None
            if status == 'success':
                print('SKIP successful:', run_dir)
                continue
            if run_dir.exists() and any(run_dir.iterdir()):
                raise RuntimeError(f'Incomplete immutable run ({status}): {run_dir}; inspect status.json and use a new EXPERIMENT name')
            print('RUN', DATASET, protocol, variant['name'], seed)
            run_experiment(cfg)

## Aggregate and verify

聚合器若发现相同 dataset/protocol/model/seed 有多个成功 run，会拒绝选择其中一个，避免挑选结果。

In [ ]:
experiment_root = RESULT_ROOT / EXPERIMENT
report_dir = REPORT_ROOT / EXPERIMENT
subprocess.run([
    'python', 'scripts/aggregate.py',
    '--results-root', str(experiment_root),
    '--output-dir', str(report_dir),
    '--reference-model', 'lassf_conv1d_concat_h64',
], check=True)

import pandas as pd
summary = pd.read_csv(report_dir / 'summary.csv')
display(summary)
subprocess.run([
    'python', 'scripts/export_table.py',
    '--summary', str(report_dir / 'summary.csv'),
    '--protocol', 'spatial_block', '--metric', 'oa',
    '--output-prefix', str(report_dir / 'main_sb_oa'),
], check=True)

In [ ]:
# Verify one run from saved prediction and ground truth.
run_dirs = sorted(path.parent for path in experiment_root.rglob('metrics.json'))
assert run_dirs, 'No successful runs found'
subprocess.run(['python', 'scripts/verify_run.py', '--run-dir', str(run_dirs[0])], check=True)

In [ ]:
# Optional gate diagnostic for the main model.
import numpy as np, matplotlib.pyplot as plt
main_runs = [p.parent for p in experiment_root.rglob('metrics.json') if 'reliability' in str(p)]
if main_runs:
    gate = np.load(main_runs[0] / 'gate.npy').reshape(-1)
    plt.hist(gate[np.isfinite(gate)], bins=30)
    plt.xlabel('Spatial branch gate weight')
    plt.ylabel('Count')
    plt.title(main_runs[0].name)
    plt.show()